# Retrieval evaluation — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def cos(a,b):
    a=np.asarray(a,float); b=np.asarray(b,float)
    return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)))
print('setup ok')

## Precision, recall, nDCG, and average precision for ranked lists
A single relevance list is enough to compute cutoff metrics, rank-discounted gain, average precision, query averaging, and a comparison between two retrieval systems.

In [ ]:
# 1) Precision@k and Recall@k on one ranked list
rel=np.array([1,0,1,1,0]); total=rel.sum(); ks=np.arange(1,6)
prec=np.array([rel[:k].mean() for k in ks]); rec=np.array([rel[:k].sum()/total for k in ks])
plt.figure(figsize=(6,3)); plt.plot(ks,prec,'-o',label='precision@k'); plt.plot(ks,rec,'-s',label='recall@k'); plt.legend(); plt.xlabel('k'); plt.title('precision and recall move differently')
assert abs(prec[2]-2/3)<1e-9 and abs(rec[2]-2/3)<1e-9

In [ ]:
# 2) DCG discounts later relevant documents
dcg=sum((2**r-1)/math.log2(i+2) for i,r in enumerate(rel)); ideal=np.array(sorted(rel,reverse=True)); idcg=sum((2**r-1)/math.log2(i+2) for i,r in enumerate(ideal)); ndcg=dcg/idcg
plt.figure(figsize=(6,3)); plt.bar(['DCG','IDCG','nDCG'],[dcg,idcg,ndcg]); plt.title('nDCG compares to the ideal ranking')
assert abs(dcg-1.9306765581)<1e-9 and abs(ndcg-0.9060254355)<1e-9

In [ ]:
# 3) Average precision averages precision at relevant ranks
hits=0; ps=[]
for i,r in enumerate(rel,1):
    if r:
        hits+=1; ps.append(hits/i)
AP=sum(ps)/total
plt.figure(figsize=(6,3)); plt.bar(['rank1','rank3','rank4'],ps); plt.ylabel('precision at hit'); plt.title('AP averages precision at each relevant hit')
assert abs(AP-0.8055555556)<1e-9

In [ ]:
# 4) Compare two systems: same recall can have different nDCG
A=np.array([1,0,1,1,0]); B=np.array([0,1,1,1,0])
def ndcg_at(x):
    dc=sum((2**r-1)/math.log2(i+2) for i,r in enumerate(x)); ii=sorted(x,reverse=True); idc=sum((2**r-1)/math.log2(i+2) for i,r in enumerate(ii)); return dc/idc
vals=[ndcg_at(A),ndcg_at(B)]
plt.figure(figsize=(6,3)); plt.bar(['system A','system B'],vals); plt.ylabel('nDCG@5'); plt.title('earlier relevance wins')
assert vals[0]>vals[1]

In [ ]:
# 5) Macro-average across queries gives each query equal weight
APs=np.array([0.8055555556,0.5,1.0]); macro=APs.mean()
plt.figure(figsize=(6,3)); plt.bar(['q1','q2','q3','MAP'],[*APs,macro]); plt.ylim(0,1.05); plt.title('mean average precision across queries')
assert abs(macro-0.7685185185)<1e-9